# 👁️ Eye-Tracking Analyse v2.0

**Multi-Line-Tracking-Analyse mit automatischer Klassifikation**

---

## 📋 Workflow

1. **CSVs hochladen** (Drag & Drop in Cell 2)
2. **Analyse ausführen** (Run All Cells)
3. **Ergebnisse herunterladen** (5 PNGs + Report)

---

## ⚙️ Setup

In [ ]:
# Dependencies installieren
!pip install pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import io
from datetime import datetime

print("✅ Dependencies installiert")

---

## 📂 CSVs hochladen

**Drag & Drop die 3 CSV-Dateien in die nächste Zelle:**
- `line1_vertical_*.csv`
- `line2_syllables_*.csv`
- `line3_words_*.csv`

In [ ]:
# CSVs hochladen
uploaded = files.upload()

# Dateien identifizieren
csv_files = {}
for filename in uploaded.keys():
    if 'line1' in filename or 'vertical' in filename:
        csv_files['line1'] = filename
    elif 'line2' in filename or 'syllable' in filename:
        csv_files['line2'] = filename
    elif 'line3' in filename or 'word' in filename:
        csv_files['line3'] = filename

print(f"\n✅ {len(csv_files)} Dateien erkannt:")
for key, fname in csv_files.items():
    print(f"  - {key}: {fname}")

if len(csv_files) != 3:
    print("\n⚠️  WARNUNG: Es sollten genau 3 CSVs hochgeladen werden!")

---

## 📊 Daten laden & validieren

In [ ]:
# Daten laden
def load_csv_skip_metadata(filename):
    """Lädt CSV und überspringt Metadaten-Header (#)"""
    return pd.read_csv(filename, comment='#')

line1_df = load_csv_skip_metadata(csv_files['line1'])
line2_df = load_csv_skip_metadata(csv_files['line2'])
line3_df = load_csv_skip_metadata(csv_files['line3'])

# Validierung
print("✅ Daten geladen:")
print(f"  Line 1 (Vertical): {len(line1_df)} Chunks")
print(f"  Line 2 (Syllables): {len(line2_df)} Silben")
print(f"  Line 3 (Words): {len(line3_df)} Wörter")

# Erste Zeilen anzeigen
print("\n📋 Line 1 (Preview):")
display(line1_df.head())

print("\n📋 Line 2 (Preview):")
display(line2_df.head())

print("\n📋 Line 3 (Preview):")
display(line3_df.head())

---

## 🔬 Statistiken berechnen

In [ ]:
# Globale Statistiken
stats = {}

# Line 1: Vertical
stats['line1_mean_FFD'] = line1_df['FFD'].mean()
stats['line1_mean_TRT'] = line1_df['TRT'].mean()
stats['line1_skip_rate'] = (line1_df['skipped'].sum() / len(line1_df)) * 100
stats['line1_mean_top_ratio'] = line1_df['top_ratio'].mean()
stats['line1_mean_fixations'] = line1_df['fixation_count'].mean()
stats['line1_mean_revisits'] = line1_df['revisits'].mean()

# Line 2: Syllables
stats['line2_mean_FFD'] = line2_df['FFD'].mean()
stats['line2_skip_rate'] = (line2_df['skipped'].sum() / len(line2_df)) * 100
stats['line2_mean_jump_distance'] = line2_df['jump_distance'].mean()
stats['line2_regressions'] = (line2_df['jump_distance'] < 0).sum()

# Line 3: Words
stats['line3_mean_FFD'] = line3_df['FFD'].mean()
stats['line3_mean_TRT'] = line3_df['TRT'].mean()
stats['line3_skip_rate'] = (line3_df['skipped'].sum() / len(line3_df)) * 100
stats['line3_mean_top_ratio'] = line3_df['top_ratio'].mean()
stats['line3_mean_left_ratio'] = line3_df['left_ratio'].mean()
stats['line3_mean_fixations'] = line3_df['fixation_count'].mean()
stats['line3_mean_revisits'] = line3_df['revisits'].mean()

# Ausgabe
print("📊 Globale Statistiken:")
print("\n🔹 Line 1 (Vertical):")
print(f"  Mean FFD: {stats['line1_mean_FFD']:.1f} ms")
print(f"  Mean TRT: {stats['line1_mean_TRT']:.1f} ms")
print(f"  Skip Rate: {stats['line1_skip_rate']:.1f}%")
print(f"  Top Ratio: {stats['line1_mean_top_ratio']:.3f}")

print("\n🔹 Line 2 (Syllables):")
print(f"  Mean FFD: {stats['line2_mean_FFD']:.1f} ms")
print(f"  Skip Rate: {stats['line2_skip_rate']:.1f}%")
print(f"  Mean Jump Distance: {stats['line2_mean_jump_distance']:.2f}")
print(f"  Regressions: {stats['line2_regressions']}")

print("\n🔹 Line 3 (Words):")
print(f"  Mean FFD: {stats['line3_mean_FFD']:.1f} ms")
print(f"  Mean TRT: {stats['line3_mean_TRT']:.1f} ms")
print(f"  Skip Rate: {stats['line3_skip_rate']:.1f}%")
print(f"  Top Ratio: {stats['line3_mean_top_ratio']:.3f}")
print(f"  Left Ratio: {stats['line3_mean_left_ratio']:.3f}")

---

## 🎯 Automatische Klassifikation

In [ ]:
# Scoring-Algorithmus
scores = {'decoder': 0, 'guesser': 0, 'compensator': 0}

# Durchschnittswerte
mean_FFD = (stats['line1_mean_FFD'] + stats['line2_mean_FFD'] + stats['line3_mean_FFD']) / 3
mean_skip_rate = (stats['line1_skip_rate'] + stats['line2_skip_rate'] + stats['line3_skip_rate']) / 3
mean_top_ratio = (stats['line1_mean_top_ratio'] + stats['line3_mean_top_ratio']) / 2
mean_fixations = (stats['line1_mean_fixations'] + stats['line3_mean_fixations']) / 2
mean_revisits = (stats['line1_mean_revisits'] + stats['line3_mean_revisits']) / 2

# DEKODIERER-INDIKATOREN
if mean_FFD > 300:
    scores['decoder'] += 2
elif mean_FFD > 250:
    scores['decoder'] += 1

if mean_fixations > 3:
    scores['decoder'] += 2
elif mean_fixations > 2:
    scores['decoder'] += 1

if 0.45 <= mean_top_ratio <= 0.55:
    scores['decoder'] += 2

if stats['line3_mean_left_ratio'] > 0.55:
    scores['decoder'] += 1

if mean_skip_rate < 10:
    scores['decoder'] += 2

# RATER-INDIKATOREN
if mean_FFD < 200:
    scores['guesser'] += 2

if mean_skip_rate > 30:
    scores['guesser'] += 3
elif mean_skip_rate > 20:
    scores['guesser'] += 2

if mean_top_ratio > 0.60:
    scores['guesser'] += 2

if stats['line2_mean_jump_distance'] > 2.0:
    scores['guesser'] += 2
elif stats['line2_mean_jump_distance'] > 1.5:
    scores['guesser'] += 1

if mean_fixations < 2:
    scores['guesser'] += 1

# KOMPENSATOR-INDIKATOREN
if mean_revisits > 1:
    scores['compensator'] += 3
elif mean_revisits > 0.5:
    scores['compensator'] += 2

if stats['line2_regressions'] > 2:
    scores['compensator'] += 2
elif stats['line2_regressions'] > 0:
    scores['compensator'] += 1

mean_TRT = (stats['line1_mean_TRT'] + stats['line3_mean_TRT']) / 2
if mean_TRT > mean_FFD * 2 and mean_FFD < 300:
    scores['compensator'] += 2

if 10 <= mean_skip_rate <= 30:
    scores['compensator'] += 1

# Klassifikation
profile = max(scores, key=scores.get)
total_score = sum(scores.values())
confidence = (scores[profile] / total_score * 100) if total_score > 0 else 0

# Interpretation
profile_names = {
    'decoder': 'DEKODIERER',
    'guesser': 'RATER',
    'compensator': 'KOMPENSATOR'
}

profile_descriptions = {
    'decoder': 'Langsames, gründliches Dekodieren (jeder Buchstabe wird gelesen)',
    'guesser': 'Schnelles Überfliegen/Raten basierend auf Wortformen',
    'compensator': 'Hoher kognitiver Aufwand durch ständige Rücksprünge und Korrekturen'
}

print("🎯 AUTOMATISCHE KLASSIFIKATION")
print("=" * 50)
print(f"\nProfil: {profile_names[profile]}")
print(f"Konfidenz: {confidence:.1f}%")
print(f"\nBegründung: {profile_descriptions[profile]}")
print("\nScores:")
for p, score in scores.items():
    print(f"  - {profile_names[p]}: {score}")

---

## 📊 Visualisierungen erstellen

In [ ]:
# 1. Vertical Distribution (Line 1)
fig, ax = plt.subplots(figsize=(12, 6))

x = range(len(line1_df))
width = 0.35

ax.bar([i - width/2 for i in x], line1_df['top_duration'], width, label='Top', color='skyblue')
ax.bar([i + width/2 for i in x], line1_df['bottom_duration'], width, label='Bottom', color='lightcoral')

ax.set_xlabel('Chunk ID', fontsize=12)
ax.set_ylabel('Duration (ms)', fontsize=12)
ax.set_title('Line 1: Vertical Distribution (Oberlängen vs. Unterlängen)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(line1_df['text'], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('line1_vertical_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualisierung 1/5 gespeichert: line1_vertical_distribution.png")

In [ ]:
# 2. Syllable Sequence (Line 2)
fig, ax = plt.subplots(figsize=(12, 6))

# Ideale Sequenz
ideal = list(range(len(line2_df)))
ax.plot(ideal, ideal, 'k--', label='Ideal (sequenziell)', linewidth=2, alpha=0.5)

# Tatsächliche Sequenz
read_order = line2_df['read_order'].fillna(-1).astype(int)
syl_ids = line2_df['syl_id'].values

# Sortiere nach read_order
sorted_indices = read_order[read_order >= 0].argsort()
sorted_syl_ids = syl_ids[read_order >= 0][sorted_indices]
sorted_order = range(len(sorted_syl_ids))

ax.plot(sorted_order, sorted_syl_ids, 'ro-', label='Tatsächlich', linewidth=2, markersize=8)

ax.set_xlabel('Lesereihenfolge (zeitlich)', fontsize=12)
ax.set_ylabel('Silben-ID (Position im Text)', fontsize=12)
ax.set_title('Line 2: Silben-Sequenz-Analyse', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('line2_syllable_sequence.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualisierung 2/5 gespeichert: line2_syllable_sequence.png")

In [ ]:
# 3. Regions Heatmap (Line 3)
regions_data = line3_df[['tl_dur', 'tr_dur', 'bl_dur', 'br_dur']].values
regions_labels = ['TL', 'TR', 'BL', 'BR']

fig, ax = plt.subplots(figsize=(12, 8))

sns.heatmap(regions_data.T, 
            annot=True, 
            fmt='.0f',
            cmap='YlOrRd',
            xticklabels=line3_df['word'],
            yticklabels=regions_labels,
            cbar_kws={'label': 'Duration (ms)'},
            ax=ax)

ax.set_title('Line 3: 4-Regionen-Heatmap (Top-Left/Right, Bottom-Left/Right)', fontsize=14, fontweight='bold')
ax.set_xlabel('Wort', fontsize=12)
ax.set_ylabel('Region', fontsize=12)

plt.tight_layout()
plt.savefig('line3_regions_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualisierung 3/5 gespeichert: line3_regions_heatmap.png")

In [ ]:
# 4. Profile Radar Chart
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))

categories = list(profile_names.values())
values = [scores['decoder'], scores['guesser'], scores['compensator']]

# Winkel für Kategorien
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
values += values[:1]  # Schließe den Kreis
angles += angles[:1]

ax.plot(angles, values, 'o-', linewidth=2, color='navy')
ax.fill(angles, values, alpha=0.25, color='navy')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, max(values) * 1.2)
ax.set_title('Leserprofil-Scores (Radar-Chart)', fontsize=14, fontweight='bold', pad=20)
ax.grid(True)

plt.tight_layout()
plt.savefig('profile_radar.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualisierung 4/5 gespeichert: profile_radar.png")

In [ ]:
# 5. Statistics Summary (4 Subplots)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Subplot 1: Skip Rates
skip_rates = [stats['line1_skip_rate'], stats['line2_skip_rate'], stats['line3_skip_rate']]
axes[0, 0].bar(['Line 1', 'Line 2', 'Line 3'], skip_rates, color=['skyblue', 'lightgreen', 'lightcoral'])
axes[0, 0].set_ylabel('Skip Rate (%)')
axes[0, 0].set_title('Skip Rates pro Zeile')
axes[0, 0].grid(axis='y', alpha=0.3)

# Subplot 2: FFD Comparison
ffds = [stats['line1_mean_FFD'], stats['line2_mean_FFD'], stats['line3_mean_FFD']]
axes[0, 1].bar(['Line 1', 'Line 2', 'Line 3'], ffds, color=['skyblue', 'lightgreen', 'lightcoral'])
axes[0, 1].set_ylabel('FFD (ms)')
axes[0, 1].set_title('First Fixation Duration (Durchschnitt)')
axes[0, 1].grid(axis='y', alpha=0.3)

# Subplot 3: Fixation Counts
fixations = [stats['line1_mean_fixations'], stats['line3_mean_fixations']]
axes[1, 0].bar(['Line 1', 'Line 3'], fixations, color=['skyblue', 'lightcoral'])
axes[1, 0].set_ylabel('Fixation Count')
axes[1, 0].set_title('Durchschnittliche Fixationen pro Wort/Chunk')
axes[1, 0].grid(axis='y', alpha=0.3)

# Subplot 4: Top/Left Ratios
ratios = [
    stats['line1_mean_top_ratio'],
    stats['line3_mean_top_ratio'],
    stats['line3_mean_left_ratio']
]
axes[1, 1].bar(['Line 1 Top', 'Line 3 Top', 'Line 3 Left'], ratios, color=['skyblue', 'lightcoral', 'gold'])
axes[1, 1].set_ylabel('Ratio')
axes[1, 1].set_title('Räumliche Fokussierung (0.5 = ausgeglichen)')
axes[1, 1].axhline(0.5, color='red', linestyle='--', linewidth=1, alpha=0.5)
axes[1, 1].grid(axis='y', alpha=0.3)

plt.suptitle('Statistik-Übersicht (Zusammenfassung)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('statistics_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualisierung 5/5 gespeichert: statistics_summary.png")

---

## 📄 Text-Report generieren

In [ ]:
# Report erstellen
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
report_filename = f'report_{timestamp}.txt'

with open(report_filename, 'w', encoding='utf-8') as f:
    f.write("=" * 60 + "\n")
    f.write("EYE-TRACKING ANALYSE-REPORT\n")
    f.write("=" * 60 + "\n\n")
    
    f.write(f"Datum: {datetime.now().strftime('%d.%m.%Y %H:%M:%S')}\n")
    f.write(f"Eingabe-Dateien:\n")
    for key, fname in csv_files.items():
        f.write(f"  - {key}: {fname}\n")
    f.write("\n" + "=" * 60 + "\n\n")
    
    # Klassifikation
    f.write("AUTOMATISCHE KLASSIFIKATION\n")
    f.write("-" * 60 + "\n\n")
    f.write(f"Profil: {profile_names[profile]}\n")
    f.write(f"Konfidenz: {confidence:.1f}%\n\n")
    f.write(f"Begründung: {profile_descriptions[profile]}\n\n")
    f.write("Scores:\n")
    for p, score in scores.items():
        f.write(f"  - {profile_names[p]}: {score}\n")
    f.write("\n" + "=" * 60 + "\n\n")
    
    # Statistiken
    f.write("GLOBALE STATISTIKEN\n")
    f.write("-" * 60 + "\n\n")
    
    f.write("Line 1 (Vertical Tracking):\n")
    f.write(f"  Mean FFD: {stats['line1_mean_FFD']:.1f} ms\n")
    f.write(f"  Mean TRT: {stats['line1_mean_TRT']:.1f} ms\n")
    f.write(f"  Skip Rate: {stats['line1_skip_rate']:.1f}%\n")
    f.write(f"  Top Ratio: {stats['line1_mean_top_ratio']:.3f}\n")
    f.write(f"  Mean Fixations: {stats['line1_mean_fixations']:.2f}\n")
    f.write(f"  Mean Revisits: {stats['line1_mean_revisits']:.2f}\n\n")
    
    f.write("Line 2 (Syllable Tracking):\n")
    f.write(f"  Mean FFD: {stats['line2_mean_FFD']:.1f} ms\n")
    f.write(f"  Skip Rate: {stats['line2_skip_rate']:.1f}%\n")
    f.write(f"  Mean Jump Distance: {stats['line2_mean_jump_distance']:.2f}\n")
    f.write(f"  Regressions: {stats['line2_regressions']}\n\n")
    
    f.write("Line 3 (Standard Word Tracking):\n")
    f.write(f"  Mean FFD: {stats['line3_mean_FFD']:.1f} ms\n")
    f.write(f"  Mean TRT: {stats['line3_mean_TRT']:.1f} ms\n")
    f.write(f"  Skip Rate: {stats['line3_skip_rate']:.1f}%\n")
    f.write(f"  Top Ratio: {stats['line3_mean_top_ratio']:.3f}\n")
    f.write(f"  Left Ratio: {stats['line3_mean_left_ratio']:.3f}\n")
    f.write(f"  Mean Fixations: {stats['line3_mean_fixations']:.2f}\n")
    f.write(f"  Mean Revisits: {stats['line3_mean_revisits']:.2f}\n\n")
    
    f.write("=" * 60 + "\n\n")
    
    # Interpretation
    f.write("INTERPRETATION\n")
    f.write("-" * 60 + "\n\n")
    
    if profile == 'decoder':
        f.write("Das Leseprofil zeigt typische Merkmale eines DEKODIERERS:\n\n")
        f.write("✓ Langsame, gründliche Verarbeitung (hohe FFD)\n")
        f.write("✓ Viele Fixationen pro Wort (gründliches Lesen)\n")
        f.write("✓ Ausgeglichene Top/Bottom-Ratio (nutzt alle Buchstaben)\n")
        f.write("✓ Geringe Skip-Rate (kaum übersprungene Wörter)\n\n")
        f.write("Empfehlung: Förderung der Leseflüssigkeit durch Übung.\n")
    elif profile == 'guesser':
        f.write("Das Leseprofil zeigt typische Merkmale eines RATERS:\n\n")
        f.write("✓ Schnelles Überfliegen (niedrige FFD)\n")
        f.write("✓ Hohe Skip-Rate (viele übersprungene Wörter/Silben)\n")
        f.write("✓ Top-betonte Fixationen (nur Oberlängen)\n")
        f.write("✓ Große Sprungweiten (chaotische Sequenz)\n\n")
        f.write("Empfehlung: Intervention mit Leetspeak/Oberlängen-Maskierung.\n")
    else:  # compensator
        f.write("Das Leseprofil zeigt typische Merkmale eines KOMPENSATORS:\n\n")
        f.write("✓ Viele Rücksprünge (hohe Revisits)\n")
        f.write("✓ Hohe TRT trotz moderater FFD (Nacharbeit)\n")
        f.write("✓ Regressionen in Silben-Sequenz\n")
        f.write("✓ Mittlere Skip-Rate (selektives Lesen)\n\n")
        f.write("Empfehlung: Training zur Reduzierung kognitiver Last.\n")
    
    f.write("\n" + "=" * 60 + "\n")
    f.write("ENDE DES REPORTS\n")
    f.write("=" * 60 + "\n")

print(f"✅ Report gespeichert: {report_filename}")

# Report auch ausgeben
with open(report_filename, 'r', encoding='utf-8') as f:
    print("\n" + f.read())

---

## 💾 Alle Ergebnisse herunterladen

In [ ]:
# Alle Dateien zum Download bereitstellen
files_to_download = [
    'line1_vertical_distribution.png',
    'line2_syllable_sequence.png',
    'line3_regions_heatmap.png',
    'profile_radar.png',
    'statistics_summary.png',
    report_filename
]

print("📥 Dateien zum Download:")
for fname in files_to_download:
    print(f"  - {fname}")
    files.download(fname)

print("\n✅ ANALYSE ABGESCHLOSSEN!")
print("\n📊 Du hast heruntergeladen:")
print("  - 5 Visualisierungen (PNG)")
print("  - 1 Text-Report (TXT)")